### Multi-Learning Problems: Multiclass, Multilabel & Multioutput

#### 1. Ye Teeno Terms Asal Me Kya Hain? (The Core Difference)
Machine Learning me jab humara target variable ($y$) thoda complex ho jata hai, toh hum use **Multi-learning problem** kehte hain. Isko 3 main categories me divide kiya gaya hai. Isko samajhne ke liye 2 cheezein dhyan me rakhni hain:
*   **Number of outputs:** Model kitni alag-alag cheezein predict kar raha hai?
*   **Number of classes/labels per output:** Har output me kitne options available hain?

#### 2. Multiclass Classification
*   **Kya hai:** Model ko **exactly 1 output** dena hai, lekin uske paas chunne ke liye **>2 options (classes)** hain.
*   **Real-life Example:** Ek fruit ki photo dekh kar batana ki wo kya hai. (Options: Apple, Banana, Orange). Answer sirf koi ek hoga.
*   **Math / Data Format:** 
    *   Target variable $y \in \{0, 1, ..., K-1\}$ jahan $K$ total classes hain.
    *   Under the hood, ye **One-Hot Encoding** use karta hai. Agar Banana (class 1) sahi hai, toh target vector = `[0, 1, 0]`.
    *   *Activation Function:* Neural networks me iske liye **Softmax** use hota hai, jo saari probabilities ka sum $1$ kar deta hai. $\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$

#### 3. Multilabel Classification
*   **Kya hai:** Model ko **>1 outputs (multiple labels)** dene hain ek hi input ke liye, aur har output binary hota hai (total #labels = 2 per output, i.e., Yes/No). Tum ek sath multiple options choose kar sakte ho.
*   **Real-life Example:** Ek movie ka genre predict karna. Ek hi movie ek sath **Action (Yes)**, **Comedy (Yes)**, aur **Horror (No)** ho sakti hai.
*   **Math / Data Format:**
    *   Target variable ek binary vector hota hai: $y \in \{0, 1\}^L$ jahan $L$ total labels hain.
    *   Example target vector: `[1, 1, 0]` (Action=1, Comedy=1, Horror=0).
    *   *Activation Function:* Isme Softmax nahi lagta, balki har label ke liye alag **Sigmoid** lagta hai. Yahan probabilities ka sum 1 hona zaroori nahi hai.

#### 4. Multioutput-Multiclass Classification
*   **Kya hai:** Ye sabse complex form hai. Model ko **>1 outputs** dene hain, aur har output ke andar **>2 options** hain.
*   **Real-life Example:** Ek gadi ki photo dekh kar do cheezein ek sath predict karna: 
    1. Gadi ka Brand (Ford, Toyota, Honda) -> >2 classes
    2. Gadi ka Color (Red, Blue, Green, Black) -> >2 classes
*   **Math / Data Format:**
    *   Target variable ek 2D array (matrix) ban jata hai.
    *   Example prediction: `[Toyota, Blue]` ya mathematically `[1, 3]`.

#### 5. Important Takeaways & Terminology 💡
*   Tumhari slide ke according, jab model ke outputs 1 se zyada hote hain (chahe wo binary ho ya multiclass), toh un dono scenarios ko broadly **"Multi-label classification models"** ya **"Multioutput models"** kaha ja sakta hai.
*   Scikit-learn me inko handle karne ke liye alag se wrappers diye gaye hain jaise `MultiOutputClassifier` ya `OneVsRestClassifier`.

---

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
import numpy as np

# ==================================================
# 1. Multiclass Classification
# ==================================================

X_train = np.random.rand(20, 5)

y_train = np.random.randint(0, 3, size=20)

# multi_class hata do
model = LogisticRegression()

model.fit(X_train, y_train)

print(model.predict(X_train[:5]))


# ==================================================
# 2. Multilabel / Multioutput Classification
# ==================================================

X_train = np.random.rand(20, 5)

y_train = np.random.randint(0, 2, size=(20, 3))

base_model = LogisticRegression()

multi_model = MultiOutputClassifier(base_model)

multi_model.fit(X_train, y_train)

print(multi_model.predict(X_train[:5]))

[0 2 0 2 2]
[[1 1 1]
 [1 1 0]
 [0 1 1]
 [1 1 0]
 [0 1 1]]


### Meta-Estimators: Solving Multi-Learning Problems

#### 1. Meta-Estimators Kya Hote Hain?
Meta-estimator ek "Wrapper" ya "Manager" ki tarah hota hai. Iska apna koi algorithm nahi hota. Ye tumhara diya hua ek basic model (jaise Logistic Regression ya SVM) leta hai, aur uski multiple copies banakar ek complex problem ko solve karta hai. Ye badi problem ko tod kar **"One model per sub-problem"** wali strategy lagata hai.

#### 2. Multiclass Meta-Estimators (`sklearn.multiclass`)
Agar tumhara target variable ek hai, par usme classes 3 ya usse zyada hain ($K > 2$), toh ye meta-estimators use hote hain:

*   **`OneVsRestClassifier` (OvR / OvA):**
    *   *Logic:* Ye har class ko baaki saari classes ke against compare karta hai. Agar $K$ classes hain, toh ye exactly **$K$ models** train karega.
    *   *Math:* Model 1 seekhega $P(y=\text{Class 1} | x)$ vs $P(y \neq \text{Class 1} | x)$. Final prediction ke time pe jis model ki probability ya confidence score sabse high hota hai, wahi class output ban jati hai: $\hat{y} = \arg\max_{k \in \{1...K\}} f_k(x)$
    *   *Kab use karein:* Ye default aur sabse fast tarika hai.

*   **`OneVsOneClassifier` (OvO):**
    *   *Logic:* Ye har do classes ke beech ek "1-on-1 match" karwata hai.
    *   *Math:* Agar $K$ classes hain, toh ye $\frac{K(K-1)}{2}$ models train karega. (Maan lo 10 classes hain, toh 45 models banenge!). Final output ke liye ye **Voting Mechanism** use karta hai — jis class ko sabse zyada "1-on-1" matches me jeet milti hai, wo final winner hoti hai.
    *   *Kab use karein:* Jab tumhara algorithm massive data pe slow ho jata hai (jaise Kernel SVM). OvO me har model sirf 2 classes ke data par train hota hai, toh chote data par multiple models train karna faster ho jata hai.

*   **`OutputCodeClassifier` (Error-Correcting Output Codes - ECOC):**
    *   *Logic:* Ye har class ko ek binary code (e.g., `1010`, `0110`) assign kar deta hai. Fir ye models ko predict karne ko kehta hai ki 1st bit kya hogi, 2nd bit kya hogi, etc.
    *   *Math:* Final prediction me model ka nikala hua code aur actual classes ke codes ka **Hamming Distance** (kitne bits match nahi ho rahe) calculate hota hai. Jiska distance sabse kam, wahi class winner. Ye prediction errors ke against bahut robust hota hai.

#### 3. Multilabel Meta-Estimators (`sklearn.multioutput`)
Jab tumhe ek sath multiple labels predict karne ho (e.g., ek movie Action **bhi** ho sakti hai aur Thriller **bhi**). Yahan target vector $y$ ek matrix hota hai jisme $L$ columns (labels) hote hain.

*   **`MultiOutputClassifier`:**
    *   *Logic:* Ye har label ke liye ek completely independent model train karta hai. Agar 5 labels hain, toh 5 models banenge.
    *   *Math:* Ye assume karta hai ki saare labels ek dusre se independent hain: $P(y_1, y_2 | x) = P(y_1 | x) \times P(y_2 | x)$.
    *   *Flaw:* Ye labels ke aapas ka relation (correlation) ignore kar deta hai.

*   **`ClassifierChain` (The Smart One):**
    *   *Logic:* Ye models ko ek "Chain" (zanjeer) me jod deta hai.
    *   *Math:* Model 1 pehle label $y_1$ ko features $X$ ki madad se predict karta hai. Model 2 dusre label $y_2$ ko predict karta hai, lekin usko features $X$ ke sath-sath **Model 1 ka prediction ($y_1$) bhi input ke roop me milta hai**. 
    $P(y_1, y_2, y_3 | x) = P(y_1 | x) \cdot P(y_2 | x, y_1) \cdot P(y_3 | x, y_1, y_2)$
    *   *Fayda:* Ye labels ke beech ke connections samajh jata hai. (Jaise agar movie "Action" (y1=1) hai, toh uske "Thriller" (y2=1) hone ke chances badh jate hain, par "Romance" ke chances ghat jate hain).
---

In [2]:

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
# (X_train aur y_train already defined hone chahiye)

# Base estimator hum Logistic Regression hi rakhenge
base_logit = LogisticRegression(solver='lbfgs')

# --- 1. Multiclass: One Vs Rest ---
# Wraps the logistic regression into an OvR strategy
ovr_classifier = OneVsRestClassifier(base_logit)
# ovr_classifier.fit(X_train, y_train_multiclass)

# --- 2. Multilabel: Classifier Chain ---
# Captures relationships between multiple target labels
chain_classifier = ClassifierChain(base_logit, order='random', random_state=42)
# chain_classifier.fit(X_train, y_train_multilabel)

### Built-in Multi-learning Support (Bina Manager Ke Kaam Karna)

#### 1. Asal Mudda Kya Hai? (What does the slide say?)
Scikit-learn library bahut smart hai. Uske lagbhag saare naye aur advanced models me pehle se hi aisi coding (built-in support) hoti hai ki wo multiple classes (Multiclass) ya multiple labels (Multilabel) ko khud hi handle kar lete hain. 
*   **Simple Example:** Agar tumhare smartphone me pehle se hi achha in-built camera hai, toh tumhe alag se DSLR camera (Meta-estimator) kharidne aur jodne ki zarurat nahi hai. Tum direct photo kheencho, phone khud handle kar lega.

#### 2. Models ki 4 Categories (Kon kaisa kaam karta hai?)
Jab hum in smart models ko multi-class data dete hain, toh ye andar hi andar apni ek fix aadat (strategy) ke hisaab se kaam karte hain:

*   **Inherently multiclass:** Ye wo models hain jo by-birth smart hain. Ye data ko todte nahi hain, balki ek hi baar me saari classes samajh lete hain. (Example: `DecisionTreeClassifier`, `RandomForestClassifier`).
*   **Multiclass as OVO (One-vs-One):** Ye models under-the-hood chupke se apne aap "1-on-1 match" karwa lete hain. Tumhe alag se `OneVsOneClassifier` lagane ki zarurat nahi padti. (Example: `SVC` - Support Vector Machine).
*   **Multiclass as OVR (One-vs-Rest):** Ye models automatically 1 class ko baaki sabse compare karne lagte hain. (Example: `LogisticRegression`).
*   **Multilabel:** Kuch models itne smart hote hain ki wo ek sath multiple tags (Yes/No) ek hi input ke liye de sakte hain bina kisi extra code ke. (Example: `KNeighborsClassifier`).

#### 3. Toh fir humne Meta-estimators kyu padhe? (Kab use karein?)
Agar in estimators me sab kuch pehle se hai, toh hum `OneVsRestClassifier` jaisi cheezein use hi kyu karte hain?
**Answer:** Jab hume model ka "Default" tarika pasand nahi aata, aur hum usse zabardasti koi aur tarika use karwana chahte hain.

*   *Example:* `SVC` by default OVO (One-vs-One) use karta hai. Lekin maan lo tumhare paas data bahut zyada hai, aur tum chahte ho ki SVC thoda fast kaam kare aur OVR (One-vs-Rest) strategy use kare. 
*   Aise me tum directly `SVC` use karne ke bajaye usko `OneVsRestClassifier` ke andar lapet (wrap) doge. 

---

In [3]:

from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier

# --- CASE 1: Default/In-built (Hassle-free) ---
# Ye automatically OVO (One-vs-One) strategy use karega
model_default = SVC()
# model_default.fit(X_train, y_train_multiclass)

# --- CASE 2: Using Meta-estimator (Forcing a new rule) ---
# Tumhe OVO pasand nahi aaya, toh tumne SVC ko Wrapper me daal diya taaki wo OVR use kare
model_forced = OneVsRestClassifier(SVC())
# model_forced.fit(X_train, y_train_multiclass)

### Ekdam Basic Bhasa Me: Models Ka Dimag Kaise Kaam Karta Hai

#### 1. Problem Kya Hai? (Hum ye kyu padh rahe hain?)
Maan lo humara algorithm ek player hai. 
Agar usko sirf ye batana ho ki samne wala **"Bot hai ya Real Player"** (sirf 2 options), toh wo aaram se bata dega. Isko **Binary Classification** kehte hain.

Lekin dikkat tab aati hai jab samne wale ke hath me gun dekh kar batana ho ki wo **"AKM hai, M416 hai, ya DP-28 hai"** (2 se zyada options). Isko **Multiclass** kehte hain. Har algorithm itna smart nahi hota ki ek baar me 3-4 cheezein dekh sake. Isiliye unke kaam karne ke alag-alag tarike (logic) hote hain:

#### 2. Kam Karne Ke Tarike (The 3 Logics)

**A. Inherently Multiclass (Paidaishi Smart)**
*   **Logic:** Ye wo pro players hain jo ek baar me gun dekh kar bol dete hain: "Bhai ye AKM hai". Inko options todne ki zarurat nahi padti. Ye ek hi sath saari guns ke features compare karte hain aur best answer de dete hain.
*   **Slide me kon hai:** 
    *   `LogisticRegression(multi_class='multinomial')`
    *   `RidgeClassifier`

**B. Multiclass as OVR (One-Vs-Rest - Jugaad wala logic)**
*   **Logic:** Ye thode beginner players hain. Inko ek sath saari guns samajh nahi aati. Toh ye **Jugaad (OVR)** lagate hain. Ye khud se Yes/No wale sawaal poochte hain:
    *   Sawaal 1: "Kya ye AKM hai, ya BAAKI KUCH AUR (Rest) hai?" -> Model bola No.
    *   Sawaal 2: "Kya ye DP-28 hai, ya BAAKI KUCH AUR hai?" -> Model bola No.
    *   Sawaal 3: "Kya ye M416 hai, ya BAAKI KUCH AUR hai?" -> Model bola Yes!
    *   Isne ek problem ko 3 chhoti problems me tod diya. Ek ko baaki sabse bhida diya.
*   **Slide me kon hai:** 
    *   `LogisticRegression(multi_class='ovr')`
    *   `SGDClassifier`
    *   `Perceptron`

**C. Multilabel (Ek sath bahut saari cheezein)**
*   **Logic:** Multiclass me gun toh ek hi hogi na (ya toh AKM hogi ya M416). Lekin Multilabel ka matlab hai ek sath multiple cheezein YES ho sakti hain. Jaise gun me **"Suppressor bhi laga hai (Yes), aur Extended Mag bhi laga hai (Yes)"**. 
*   **Slide me kon hai:**
    *   `RidgeClassifier` (Ye algorithm is tarah ke data ko bhi handle kar leta hai).

#### 3. Mujhe Code Me Kya Karna Hai?
Tumhe bas ye yaad rakhna hai ki agar tum `LogisticRegression` use kar rahe ho, toh bracket ke andar tum usko bata sakte ho ki kounsa logic lagana hai:
*   `multi_class='multinomial'` likhoge toh wo pro player ban jayega (sab ek sath sochega).
*   `multi_class='ovr'` likhoge toh wo jugaad lagayega (ek-ek karke Yes/No sochega).

---

### Logistic aur Ridge me hi OVR/Multinomial kyu lagana padta hai?

#### 1. Asli Wajah (The Root Cause)
Machine learning me kuch algorithms **paidaishi (by birth) sirf "Yes" ya "No"** bolna jaante hain. 
**Logistic Regression** aur **Ridge Classifier** ka core formula is tarah banaya gaya tha ki wo sirf ek sidhi line (straight line) khinchte hain data ke beech me. Ek line ke sirf 2 side hote hain—Left ya Right (Class 0 ya Class 1, Spam ya Not Spam). 

Matlab, inki **factory fit aadat sirf Binary (2 options)** solve karne ki hai.

#### 2. Fir Multiclass (3 ya 3 se zyada options) kaise karein?
Ab tumhare paas data aa gaya jisme 5 classes hain (jaise Apple, Mango, Banana, Orange, Grapes). 
Lekin tumhara Logistic Regression toh kehta hai: *"Bhai, mujhe toh sirf 2 cheezon me line khinchna aata hai, main 5 ko kaise alag karu?"*

Is bechare Binary classifier se Multiclass ka kaam nikalwane ke liye engineers ne 2 raste (solutions) nikale:

*   **Rasta 1: Usse wahi kaam karwao jo use aata hai (OVR - One Vs Rest)**
    *   Kyunki Logistic Regression ko sirf Yes/No aata hai, humne usko bola: "Theek hai, tu pehle sirf Apple aur 'Bache hue baaki sab' (Rest) ke beech line khinch (Yes/No)."
    *   Fir humne bola: "Ab Mango aur 'Baaki sab' ke beech line khinch."
    *   Yani hum uski basic aadat (Yes/No) ka fayda utha kar baar-baar usse sawaal poochte hain. Yahi wajah hai ki hum `LogisticRegression(multi_class='ovr')` likhte hain. Hum usko trick kar rahe hain!

*   **Rasta 2: Uska Engine Upgrade kar do (Multinomial)**
    *   Dusra tareeka ye tha ki hum Logistic Regression ke internal maths (engine) me thoda modification kar dein. Humne uska Yes/No wala purana purza nikal kar usme ek naya purza (jisko math me **Softmax** bolte hain) fit kar diya. 
    *   Ab ye upgraded Logistic Regression ek baar me sabko dekh kar percentage bata sakta hai (Apple 70%, Mango 20%, Banana 10%). Yahi `multi_class='multinomial'` hai.

#### 3. Toh kya baaki models me ye problem nahi aati?
Bilkul nahi! Kuch models naturally multi-class hote hain. 
Jaise **Decision Tree**. Decision tree koi line nahi khinchta, wo conditional rules banata hai (Agar rang laal hai aur aakar gol hai -> Apple; Agar lamba hai aur peela hai -> Banana). Uske rules kitni bhi classes ke liye ban sakte hain. Usme `ovr` ya `multinomial` set karne ka chakkar hi nahi hota.

#### Summary Conclusion 💡
Hum Logistic Regression aur Ridge/SGD classifiers me `OVR` ya `Multinomial` parameter isliye specify karte hain kyunki **ye models naturally sirf Binary (2 classes) ke liye bane the**. Unse zyada classes ka kaam nikalwane ke liye hume unko batana padta hai ki: *"Bhai, jugaad (OVR) lagana hai, ya upgraded engine (Multinomial) use karna hai?"*

### MultiClass-Classification

In [14]:
from sklearn.preprocessing import LabelBinarizer
from sklearn.utils.multiclass import type_of_target
import numpy as np

y = np.array(["apple" , "pear" , "apple" , "orange"])
y_dense = LabelBinarizer().fit_transform(y)
print(y_dense)

print(type_of_target(y))

[[1 0 0]
 [0 0 1]
 [1 0 0]
 [0 1 0]]
multiclass
